In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import Word2Vec

# 1. Cấu hình đường dẫn
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
ARTICLES_FILE = BASE_PATH + "processed/articles_processed.parquet"
CUSTOMERS_FILE = BASE_PATH + "processed/customers_processed.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Khởi tạo Spark
spark = SparkSession.builder \
    .appName("HM_Metadata_Recall_W8") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

# 3. Đọc dữ liệu
transactions = spark.read.parquet(INPUT_FILE)
articles = spark.read.parquet(ARTICLES_FILE).withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))
customers = spark.read.parquet(CUSTOMERS_FILE)

# 4. Cấu hình mốc thời gian: Tuần 8 là 7 ngày cuối cùng trong dữ liệu
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
test_start_date = max_date - datetime.timedelta(days=7)

# Lấy TOÀN BỘ lịch sử TRƯỚC Tuần 8 (đã bao gồm Tuần 7) để tạo Profile khách hàng
train_data = transactions.filter(F.col("t_dat") < F.lit(test_start_date)) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

print(f"✅ Spark Ready! Đang tạo ứng viên cho Tuần 8 dựa trên lịch sử đến hết ngày {test_start_date}")

Mounted at /content/drive
✅ Spark Ready! Đang tạo ứng viên cho Tuần 8 dựa trên lịch sử đến hết ngày 2020-09-14 17:00:00


In [ ]:
# 1. Tìm Category hot nhất sàn (cho khách mới hoàn toàn)
top_global_cat = articles.join(train_data, "article_id") \
    .groupBy("product_group_name").count() \
    .orderBy(F.desc("count")).limit(1).collect()[0][0]

# 2. Tìm Category yêu thích nhất của từng khách hàng cũ
user_fav_cat = train_data.join(F.broadcast(articles.select("article_id", "product_group_name")), "article_id") \
    .groupBy("customer_id", "product_group_name").count()

user_profile = user_fav_cat.withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("count")))) \
    .filter(F.col("rn") == 1).select("customer_id", "product_group_name")

# 3. Gán Category cho toàn bộ tập khách hàng (xử lý Cold-start)
user_profile_full = customers.select("customer_id").join(user_profile, "customer_id", "left") \
    .fillna({"product_group_name": top_global_cat})

# 4. Tìm Top 15 món Trending của mỗi Category dựa trên dữ liệu Tuần 7
# (Tuần 7 là tuần ngay trước Tuần 8)
val_start_date = test_start_date - datetime.timedelta(days=7)
top_trending_by_cat = train_data.filter(F.col("t_dat") >= F.lit(val_start_date)) \
    .join(F.broadcast(articles.select("article_id", "product_group_name")), "article_id") \
    .groupBy("product_group_name", "article_id").count() \
    .withColumn("rn", F.row_number().over(Window.partitionBy("product_group_name").orderBy(F.desc("count")))) \
    .filter(F.col("rn") <= 15).select("product_group_name", "article_id")

# Tạo ứng viên Agg & Seasonal cho Tuần 8
cand_agg_seasonal = user_profile_full.join(F.broadcast(top_trending_by_cat), "product_group_name") \
    .select("customer_id", "article_id", F.lit(1).alias("priority"))

print(f"✅ Đã tạo xong ứng viên Trending từ dữ liệu Tuần 7.")

✅ Đã tạo xong ứng viên Trending từ dữ liệu Tuần 7.


In [ ]:
# --- 1. Huấn luyện Word2Vec dựa trên lịch sử toàn bộ 7 tuần ---
sequences = train_data.groupBy("customer_id").agg(F.collect_list("article_id").alias("item_list")).filter(F.size("item_list") >= 2)
w2v = Word2Vec(vectorSize=16, minCount=2, inputCol="item_list", outputCol="model_vector")
w2v_model = w2v.fit(sequences)
item_vectors = w2v_model.getVectors()

# Lấy món đồ cuối cùng khách đã mua tính đến hết Tuần 7
window_last = Window.partitionBy("customer_id").orderBy(F.desc("t_dat"))
last_items = train_data.withColumn("rn", F.row_number().over(window_last)) \
    .filter(F.col("rn") == 1).select("customer_id", "article_id")

cand_similarity = last_items.join(item_vectors, last_items.article_id == item_vectors.word) \
    .select("customer_id", "article_id", F.lit(0).alias("priority"))

# --- 2. Cùng mẫu khác màu (Priority 2) ---
last_items_7 = last_items.withColumn("p_code", F.substring(F.col("article_id"), 1, 7))
all_variants = articles.select(F.col("article_id").alias("v_id"), F.substring(F.col("article_id"), 1, 7).alias("p_code"))

cand_same_model = last_items_7.join(F.broadcast(all_variants), "p_code") \
    .filter(F.col("article_id") != F.col("v_id")) \
    .select("customer_id", F.col("v_id").alias("article_id"), F.lit(2).alias("priority"))

print("✅ Đã tạo xong ứng viên Similarity và Biến thể cùng mẫu cho Tuần 8.")

✅ Đã tạo xong ứng viên Similarity và Biến thể cùng mẫu cho Tuần 8.


In [ ]:
# 1. Gộp 3 nguồn Metadata
all_meta = cand_similarity.union(cand_agg_seasonal).union(cand_same_model)

# 2. Xếp hạng và áp dụng hạn ngạch cho từng loại ưu tiên
window_limit = Window.partitionBy("customer_id", "priority").orderBy(F.lit(1))
all_meta_refined = all_meta.withColumn("internal_rn", F.row_number().over(window_limit)) \
    .filter(
        ((F.col("priority") == 0) & (F.col("internal_rn") <= 12)) | # Word2Vec lấy 12
        ((F.col("priority") == 1) & (F.col("internal_rn") <= 10)) | # Trending lấy 10
        ((F.col("priority") == 2) & (F.col("internal_rn") <= 8))    # Biến thể lấy 8
    )

# 3. Chọn 30 ứng viên duy nhất cuối cùng
final_window = Window.partitionBy("customer_id").orderBy("priority", "internal_rn")
meta_candidates_final = all_meta_refined.dropDuplicates(["customer_id", "article_id"]) \
    .withColumn("rank", F.row_number().over(final_window)) \
    .filter(F.col("rank") <= 30) \
    .groupBy("customer_id").agg(F.collect_list("article_id").alias("meta_candidates"))

# 4. LƯU FILE VỚI HẬU TỐ _W8
meta_candidates_final.write.mode("overwrite").parquet(OUTPUT_DIR + "meta_candidates_pro_W8.parquet")

print(f"🏆 XUẤT FILE THÀNH CÔNG: meta_candidates_pro_W8.parquet")
print(f"📊 Tổng số khách hàng bao phủ cho Tuần 8: {meta_candidates_final.count():,}")

🏆 XUẤT FILE THÀNH CÔNG: meta_candidates_pro_W8.parquet
📊 Tổng số khách hàng bao phủ cho Tuần 8: 1,371,980


In [ ]:
# Giao dịch thực tế trong Tuần 8
ground_truth = transactions.filter(F.col("t_dat") >= F.lit(test_start_date)) \
    .select("customer_id", F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id"))

actual_counts = ground_truth.groupBy("customer_id").count().withColumnRenamed("count", "actual_cnt")

print(f"✅ Đã chuẩn bị xong Ground Truth cho {actual_counts.count():,} khách hàng trong Tuần 8.")

✅ Đã chuẩn bị xong Ground Truth cho 75,481 khách hàng trong Tuần 8.


In [ ]:
def evaluate_source_w8(source_df, limit=10):
    window_spec = Window.partitionBy("customer_id").orderBy("priority")
    cand_exploded = source_df.withColumn("rn", F.row_number().over(window_spec)) \
                             .filter(F.col("rn") <= limit).select("customer_id", "article_id")

    hits = ground_truth.join(cand_exploded, ["customer_id", "article_id"], "inner") \
                       .groupBy("customer_id").count().withColumnRenamed("count", "hit_cnt")

    recall_df = actual_counts.join(hits, "customer_id", "left").fillna(0)
    return recall_df.select(F.avg(F.col("hit_cnt") / F.col("actual_cnt"))).collect()[0][0]

print("🚀 Đang phân tích hiệu quả Metadata trên tập dữ liệu thực tế Tuần 8...")
recall_total = evaluate_source_w8(all_meta.dropDuplicates(["customer_id", "article_id"]), limit=30)

print("\n" + "="*55)
print(f"🔥 KẾT QUẢ RECALL@30 TRÊN TUẦN 8: {recall_total:.6f}")
print("="*55)

🚀 Đang phân tích hiệu quả Metadata trên tập dữ liệu thực tế Tuần 8...

🔥 KẾT QUẢ RECALL@30 TRÊN TUẦN 8: 0.048391
